# Pipeline LoRA (Generador 1) — Google Colab

Corre `finetuning/entrenar_lora.py` sobre GPU gratuita de Colab. Tiene
dos partes independientes en este notebook:

- **Parte A — prueba de humo** (celdas 1-6): ya corrida y documentada
  (`finetuning/lora_prueba/`, `finetuning/prueba_loss.md`, Sesión 14).
  Déjala si solo quieres repetirla; si ya vas directo al entrenamiento
  completo, corre igual las celdas 1-3 (GPU, clonar, instalar) y salta
  a la Parte B.
- **Parte B — entrenamiento completo** (celdas 7+): TODOS los ejemplos
  de `train.json`, con validación sobre `val.json` y selección del
  mejor checkpoint (Sesión 19).

**Se probó primero en la máquina local del equipo (sin GPU CUDA) y un
solo paso de entrenamiento tardaba 80-95 minutos** (a las 10 horas solo
se había completado el 5%, 8 de 150 pasos) — por eso todo esto corre en
Colab. Ver `BITACORA.md` (Sesión 14) para el detalle completo.

**Antes de correr nada**: `Entorno de ejecución` → `Cambiar tipo de
entorno de ejecución` → acelerador por hardware = **GPU** (T4 alcanza).

In [ ]:
# 1. Confirmar que hay GPU asignada (si esto falla, revisa el paso de
# arriba: Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU)
!nvidia-smi

In [ ]:
# 2. Clonar el repo (público, no hace falta token). Usa rutas
# absolutas y solo clona si hace falta, para que sea seguro volver a
# correr esta celda sin terminar con un clon anidado dentro de otro.
import os
%cd /content
if not os.path.isdir("/content/traductor-jerga-dialectal-slm"):
    !git clone https://github.com/Anderfg13/traductor-jerga-dialectal-slm.git
%cd /content/traductor-jerga-dialectal-slm

In [ ]:
# 3. Instalar solo lo que hace falta para este script (NO usar
# requirements.txt completo: Colab ya trae torch con CUDA preinstalado
# y reinstalarlo desde requirements.txt podría romper esa integración;
# tampoco hace falta mergekit/bitsandbytes/fastapi para esta prueba).
# Se desinstala torchao: Colab lo trae preinstalado en una versión
# vieja (0.10.0) y peft revisa esa versión aunque no lo usemos para
# nada aquí (no cuantizamos con torchao) -> sin él, peft simplemente
# se salta esa revisión rota.
!pip install -q transformers peft accelerate python-dotenv
!pip uninstall -y -q torchao

In [ ]:
# 4. Correr el entrenamiento de prueba (carga de datos -> tokenizacion ->
# entrenamiento LoRA -> guardado del adaptador -> recarga desde disco ->
# inferencia de prueba). Con GPU debería tardar minutos, no horas.
!python finetuning/entrenar_lora.py

In [ ]:
# 5. Ver la curva de pérdida directamente aquí, sin esperar la descarga
!cat finetuning/lora_prueba/loss_log.json

In [ ]:
# 6. Empaquetar y descargar los resultados (adaptador + loss_log +
# salidas de prueba) para traerlos al repo local y hacer commit
!zip -rq lora_prueba.zip finetuning/lora_prueba
from google.colab import files
files.download("lora_prueba.zip")

## Parte B — Entrenamiento completo (Sesión 19)

Requiere haber corrido ya las celdas 1-3 de arriba (GPU, clonar repo,
instalar dependencias) en esta misma sesión de Colab — no hace falta
repetir la Parte A.

Entrena sobre TODOS los ejemplos de `train.json` (189 para el
Generador 1), validando sobre `val.json` (24 ejemplos) al final de
cada época. Si la pérdida de validación deja de mejorar durante 2
épocas seguidas, el entrenamiento se detiene ahí y guarda el MEJOR
checkpoint (no el último) — así no hace falta vigilar manualmente si
el modelo empieza a sobreajustar.

In [ ]:
# 7. Entrenamiento completo con validación (puede tardar bastante más
# que la prueba de humo -- 189 ejemplos x hasta 10 épocas, con eval
# cada época -- pero sigue siendo minutos/pocas horas con GPU, no días).
!python finetuning/entrenar_lora.py --todos

In [ ]:
# 8. Ver la curva de pérdida (entrenamiento y validación) directamente
# aquí, sin esperar la descarga
!cat finetuning/checkpoints/generador1/loss_log.json

In [ ]:
# 9. Empaquetar y descargar el checkpoint final (adaptador + loss_log +
# salidas de prueba) para traerlo al repo local y hacer commit. Los
# checkpoints INTERMEDIOS del Trainer no están acá -- entrenar_lora.py
# los guarda en un directorio temporal de la VM de Colab, fuera del
# repo, así que este .zip solo trae el resultado final ya elegido.
!zip -rq checkpoints_generador1.zip finetuning/checkpoints/generador1
from google.colab import files
files.download("checkpoints_generador1.zip")